In [1]:
import os
import numpy as np
import pandas as pd
import pyedflib
from pyedflib import highlevel
from scipy.signal import resample
from sklearn.utils import shuffle
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
root = 'CAUEEG/'
# participants file path
annotation_path = os.path.join(root, 'annotation.xlsx')
annotations = pd.read_excel(annotation_path)
annotations

,serial,age,dementia,ad,load,eoad,vd,sivd,ad_vd_mixed,mci,...,ftd,bvftd,language_ftd,semantic_aphasia,non_fluent_aphasia,parkinson_synd,parkinson_disease,parkinson_dementia,nph,tga
0,1,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,78,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,1384,57,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1375,1385,77,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1376,1386,80,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1377,1387,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Labels

In [5]:
conditions = [
    annotations["dementia"] == 1,         # condition 1：Dementia: Alzheimer's Disease, Vascular Dementia, Parkinson's Dementia 
    annotations["mci"] == 1,        # condition 2：Mild Cognitive Impairment
    annotations["normal"] == 1      # condition 3：Normal control
]
choices = [1, 2, 0]        # corresponding labels, De=1, MCI=2, NC=0

# np.select, if no condition is met, default to 3
annotations["label"] = np.select(conditions, choices, default=3)
annotations

,serial,age,dementia,ad,load,eoad,vd,sivd,ad_vd_mixed,mci,...,bvftd,language_ftd,semantic_aphasia,non_fluent_aphasia,parkinson_synd,parkinson_disease,parkinson_dementia,nph,tga,label
0,1,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
1,2,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,3,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,4,78,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,5,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,1384,57,1.0,1.0,NaN,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1375,1385,77,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1376,1386,80,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1377,1387,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2


## Features

In [6]:
f = pyedflib.EdfReader("CAUEEG/signal/edf/00001.edf")
channels = f.getSignalLabels()
f.close()

In [7]:
channels

['Fp1-AVG',
 'F3-AVG',
 'C3-AVG',
 'P3-AVG',
 'O1-AVG',
 'Fp2-AVG',
 'F4-AVG',
 'C4-AVG',
 'P4-AVG',
 'O2-AVG',
 'F7-AVG',
 'T3-AVG',
 'T5-AVG',
 'F8-AVG',
 'T4-AVG',
 'T6-AVG',
 'FZ-AVG',
 'CZ-AVG',
 'PZ-AVG',
 'EKG',
 'Photic']

In [8]:
# 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
li_std_ch = ['Fp1-AVG', 'Fp2-AVG', 'F7-AVG', 'F3-AVG', 'FZ-AVG', 'F4-AVG', 'F8-AVG', 'T3-AVG', 'C3-AVG', 'CZ-AVG', 'C4-AVG', 'T4-AVG', 'T5-AVG', 'P3-AVG', 'PZ-AVG', 'P4-AVG', 'T6-AVG', 'O1-AVG', 'O2-AVG']

In [9]:
def generate_mask_from_events(json_path, total_samples, sample_rate=256, artifact_keywords=None):
    """
    Generate a boolean mask for valid EEG samples based on events.
    Marks valid recording periods and excludes artifacts + HV intervals.

    Parameters
    ----------
    json_path : str
        Path to the event JSON file.
    total_samples : int
        Total number of EEG samples.
    sample_rate : int
        Sampling rate in Hz (default: 256).
    artifact_keywords : list of str, optional
        Keywords to detect noisy artifacts (default: common EEG noise terms).

    Returns
    -------
    mask : np.ndarray of bool
        Boolean array indicating valid EEG data samples.
    """

    if artifact_keywords is None:
        artifact_keywords = [
            'artifact', 'blink', 'eye', 'noise', 'movement', 'move', 'cough',
            'muscle', 'chewing', 'swallow', 'talk', 'sleep', 'sweating', 'jerking', 'drowsy'
        ]

    with open(json_path, 'r') as f:
        events = json.load(f)

    mask = np.zeros(total_samples, dtype=bool)

    # Step 1: Mark valid recording periods
    is_recording = False
    segment_start = None
    for timestamp, label in events:
        label_lower = label.lower()
        if label_lower in ("start recording", "recording resumed"):
            segment_start = timestamp
            is_recording = True
        elif label_lower == "paused" and is_recording:
            segment_end = timestamp
            mask[segment_start:min(segment_end, total_samples)] = True
            is_recording = False
            segment_start = None

    # Step 2: Remove artifact regions (±1s)
    for timestamp, label in events:
        label_lower = label.lower()
        if any(key in label_lower for key in artifact_keywords):
            start = max(0, timestamp - int(1 * sample_rate))
            end = min(total_samples, timestamp + int(1 * sample_rate))
            mask[start:end] = False

    return mask

In [10]:
def data_preprocessing(
    eeg_data: np.ndarray,
    sfreq: float,
    ch_names: list,
    resample_sfreq: float = 200.0,
    verbose=True
):
    """
    Clean EEG data using bandpass filtering, percentile-based bad channel detection,
    ICA + ICLabel artifact removal, resampling, re-referencing, epoching, and z-score normalization.

    Args:
        eeg_data (np.ndarray): EEG data, shape (T, C).
        sfreq (float): Original sampling frequency.
        ch_names (list): List of channel names.
        resample_sfreq (float): Target sampling frequency.
        verbose (bool): Verbose output.

    Returns:
        np.ndarray: Cleaned, normalized EEG data, shape (n_epochs, time_steps, channels).
    """
    # 1. Construct MNE Raw object
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=['eeg'] * len(ch_names))
    raw = mne.io.RawArray(eeg_data.T, info)

    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1020'))
    if verbose:
        print("✔ Montage set: 'standard_1020'.")

    #  Bandpass Filter (0.5–45 Hz)
    raw.filter(l_freq=0.5, h_freq=45.0, verbose=False)
    if verbose:
        print("✔ Bandpass filter applied (0.5–45 Hz).")
        
    return raw.get_data().T, raw.n_times
        

def eeg_data(std_edf_path, event_path, li_std_ch, target_freq=128):
    signals, signal_headers, _ = highlevel.read_edf(std_edf_path, ch_names=li_std_ch)
    freq = signal_headers[0]['sample_frequency']
    print("Original frequency ", freq)
    data = signals.T
    print("Raw data shape ", data.shape)
    mask = generate_mask_from_events(event_path, data.shape[0], freq)
    data = data[mask]
    print("Cleaned data shape ", data.shape)
    return data

In [11]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [12]:
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
n_channels = len(li_std_ch)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP)

file_root_path = 'CAUEEG/signal/edf/'
event_root_path = 'CAUEEG/event/'
for file, event in zip(os.listdir(file_root_path), os.listdir(event_root_path)):
    if file.endswith('.edf'):
        # file example: '00001.edf'
        sub = file[:-4]  # subject ID
        print(f"[PASS 1] Subject: {file}")
        file_path = os.path.join(file_root_path, file)
        event_path = os.path.join(event_root_path, event)
        for fs in SAMPLE_RATE_LIST:
            data = eeg_data(file_path, event_path, li_std_ch, fs)
            T_raw = data.shape[0]
            print("  Original length T_raw =", T_raw)
            # resampled length if each trial
            original_fs = 200.0  # original sampling rate
            T_res = int(T_raw * fs / original_fs)
            # compute number of segments
            n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
            subject_segment_counts[sub][fs] += n_seg
            N_total += n_seg
            print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
            print("----------------------------------------")
        print("----------------End of Subject-----------------\n")

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200
[PASS 1] Subject: 00001.edf
Original frequency  200.0
Raw data shape  (145600, 19)
Cleaned data shape  (133400, 19)
  Original length T_raw = 133400
fs=200Hz: T_res=133400, STEP=200, n_seg=666
----------------------------------------
Original frequency  200.0
Raw data shape  (145600, 19)
Cleaned data shape  (133400, 19)
  Original length T_raw = 133400
fs=100Hz: T_res=66700, STEP=200, n_seg=332
----------------------------------------
Original frequency  200.0
Raw data shape  (145600, 19)
Cleaned data shape  (133400, 19)
  Original length T_raw = 133400
fs=50Hz: T_res=33350, STEP=200, n_seg=165
----------------------------------------
----------------End of Subject-----------------

[PASS 1] Subject: 00002.edf
Original frequency  200.0
Raw data shape  (208600, 19)
Cleaned data shape  (191944, 19)
  Original length T_raw = 191944
fs=200Hz: T_res=191944, STEP=200, n_seg=958
----------------------------------------
Original frequency  200.0
Raw data 

In [16]:
output_root = os.path.join('Processed', sub_folder_path, 'CAUEEG')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\CAUEEG\X.dat
y path: Processed\L400\CAUEEG\y.dat


## PASS 2: Process and store segments into memmap

In [17]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
for file, event in zip(os.listdir(file_root_path), os.listdir(event_root_path)):
    if file.endswith('.edf'):
        # file example: '00001.edf'
        sub = file[:-4]  # subject ID
        file_path = os.path.join(file_root_path, file)
        event_path = os.path.join(event_root_path, event)
        pid = int(sub)
        label = annotations.loc[annotations['serial'] == pid]['label'].values[0]
    
        total_seg_sub = sum(subject_segment_counts[sub][fs] for fs in SAMPLE_RATE_LIST)
        if total_seg_sub == 0:
            continue
        print(f"[PASS 2] Subject: {sub}, expected total segments={total_seg_sub}")
    
        # load full data
        original_fs = 200.0  # original sampling rate
        data_raw = eeg_data(file_path, event_path, li_std_ch, original_fs).astype('float32')  # (T, C)
        total_seconds_all += data_raw.shape[0] / original_fs
        print("raw shape:", data_raw.shape)

        for fs in SAMPLE_RATE_LIST:
            # resample to target fs
            data = resample_time_series(data_raw, original_fs, fs)
            T_res, _ = data.shape
            print(f"fs={fs}Hz: resampled shape={data.shape}")

            # overlapped sliding window with fixed STEP (in timestamps)
            starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
            print(f"fs={fs}Hz: segments={len(starts)}")

            for s in starts:
                if cur >= N_total:
                    raise RuntimeError("Exceeded predicted N_total.")

                X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                y_mm[cur, 0] = float(label)       # label
                y_mm[cur, 1] = float(pid)         # subject ID
                y_mm[cur, 2] = float(fs)          # sample rate (scale)
                cur += 1
    print("-------------------------------------\n")

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

[PASS 2] Subject: 00001, expected total segments=1163
Original frequency  200.0
Raw data shape  (145600, 19)
Cleaned data shape  (133400, 19)
raw shape: (133400, 19)
fs=200Hz: resampled shape=(133400, 19)
fs=200Hz: segments=666
fs=100Hz: resampled shape=(66700, 19)
fs=100Hz: segments=332
fs=50Hz: resampled shape=(33350, 19)
fs=50Hz: segments=165
-------------------------------------

[PASS 2] Subject: 00002, expected total segments=1674
Original frequency  200.0
Raw data shape  (208600, 19)
Cleaned data shape  (191944, 19)
raw shape: (191944, 19)
fs=200Hz: resampled shape=(191944, 19)
fs=200Hz: segments=958
fs=100Hz: resampled shape=(95972, 19)
fs=100Hz: segments=478
fs=50Hz: resampled shape=(47986, 19)
fs=50Hz: segments=238
-------------------------------------

[PASS 2] Subject: 00003, expected total segments=1078
Original frequency  200.0
Raw data shape  (130200, 19)
Cleaned data shape  (123600, 19)
raw shape: (123600, 19)
fs=200Hz: resampled shape=(123600, 19)
fs=200Hz: segments=61

## Load and check the processed data

In [18]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 1773814
  T = 400
  C = 19
  X_path = Processed\L400\CAUEEG\X.dat
  y_path = Processed\L400\CAUEEG\y.dat
-------------------------------------
Subject ID 001: X shape=(1163, 400, 19), y shape=(1163, 3)
Subject ID 002: X shape=(1674, 400, 19), y shape=(1674, 3)
Subject ID 003: X shape=(1078, 400, 19), y shape=(1078, 3)
Subject ID 004: X shape=(1331, 400, 19), y shape=(1331, 3)
Subject ID 005: X shape=(1387, 400, 19), y shape=(1387, 3)
Subject ID 006: X shape=(1310, 400, 19), y shape=(1310, 3)
Subject ID 007: X shape=(1435, 400, 19), y shape=(1435, 3)
Subject ID 008: X shape=(784, 400, 19), y shape=(784, 3)
Subject ID 010: X shape=(1691, 400, 19), y shape=(1691, 3)
Subject ID 011: X shape=(1456, 400, 19), y shape=(1456, 3)
Subject ID 012: X shape=(1190, 400, 19), y shape=(1190, 3)
Subject ID 013: X shape=(1197, 400, 19), y shape=(1197, 3)
Subject ID 014: X shape=(1061, 400, 19), y shape=(1061, 3)
Subject ID 015: X shape=(1426, 400, 19), y shape=(1426, 3)
Subject ID 016: X sha